# NBA Longitudinal Performance Analysis

## Problem Statement

#### How stable are individual NBA performance metrics across seasons, and which stats are most predictive of future performance?

I've been an avid sports fan my entire life and going to university sparked a passion for college and professional basketball. I love looking at statistics to predict the next games' outcomes, and this question allows me to learn more about the sport in the past three decades while putting my programming and statistics skills to use. There are lots of stats to consider in basketball, from points to assists to rebounds to minutes played, so knowing which are the most important is an interesting research question that I would love to find more about.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Dataset

This data set contains over 20 years of data on each player who has been a part of an NBA teams' roster. It contains demographics such as age, height, weight, and place of birth as well as game statistics like average points, assists, rebounds, and games played. It also has draft year, round, team played for, and other biographicals.

Dataset credits to Kaggle user Justinas Cirtautas.

In [2]:
nba_data = pd.read_csv("../data/raw/nba_player_stats.csv")

nba_data.head()

,Unnamed: 0,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,...,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season
12839,12839,Joel Embiid,PHI,29.0,213.36,127.005760,Kansas,Cameroon,2014,1,...,33.1,10.2,4.2,8.8,0.057,0.243,0.370,0.655,0.233,2022-23
12840,12840,John Butler Jr.,POR,20.0,213.36,86.182480,Florida State,USA,Undrafted,Undrafted,...,2.4,0.9,0.6,-16.1,0.012,0.065,0.102,0.411,0.066,2022-23
12841,12841,John Collins,ATL,25.0,205.74,102.511792,Wake Forest,USA,2017,1,...,13.1,6.5,1.2,-0.2,0.035,0.180,0.168,0.593,0.052,2022-23
12842,12842,Jericho Sims,NYK,24.0,208.28,113.398000,Texas,USA,2021,2,...,3.4,4.7,0.5,-6.7,0.117,0.175,0.074,0.780,0.044,2022-23
12843,12843,JaMychal Green,GSW,33.0,205.74,102.965384,Alabama,USA,Undrafted,Undrafted,...,6.4,3.6,0.9,-8.2,0.087,0.164,0.169,0.650,0.094,2022-23


In [18]:
# show number of rows and columns, respectively
nba_data.shape

(12844, 22)

In [19]:
# all column names in the dataframe
nba_data.columns

Index(['Unnamed: 0', 'player_name', 'team_abbreviation', 'age',
       'player_height', 'player_weight', 'college', 'country', 'draft_year',
       'draft_round', 'draft_number', 'gp', 'pts', 'reb', 'ast', 'net_rating',
       'oreb_pct', 'dreb_pct', 'usg_pct', 'ts_pct', 'ast_pct', 'season'],
      dtype='object')

Player identifier: ```"player_name"```
Season variable: ```"season"```
Season time variable: ```"gp"```

In [27]:
# number of unique players in the NBA since the 1997 season
nba_data["player_name"].nunique()

2551

In [34]:
nba_data.mean(numeric_only=True)

Unnamed: 0       6421.500000
age                27.045313
player_height     200.555097
player_weight     100.263279
gp                 51.154158
pts                 8.212582
reb                 3.558486
ast                 1.824681
net_rating         -2.226339
oreb_pct            0.054073
dreb_pct            0.140646
usg_pct             0.184641
ts_pct              0.513138
ast_pct             0.131595
dtype: float64



## Unit of Observation

Players' season composite z-score of box statistics (points, rebounds, assists, etc.)

## Outcome variable

Players' future season statistics (" ")

This is a good measurement because it includes statistics that typically indicate a good player (scores a lot, is a team player, aggressive in the paint). We can then use the trends to predict whether a player will have a similar season the next year, should he play. We can also analyze what statistics early will lead to a long career in the NBA, or if early-career burnout is real.

## Key Assumptions

More games played creates more opportunity for players to learn and shine; better teams can inflate individual stats

## What Could Go Wrong

Position changes; survivorship bias; mid-season injuries; lack of defensive statistics

## What this dataset allows me to do:

I can track the offensive statistics of players across seasons and determine what statistics indicate a longer season in the NBA.
I can analyze trends within the statistics across seasons to find patterns between players, positions, and teams.
I can identify revolutions such as the three point revolution due to dynamic changes in box stats across the league.

## What is this dataset does not allow me to do:

I can't track defensive statistics.
I won't know if a player leaves the NBA for a purpose other than performance issues such as injury, bad team chemistry, inactivity, etc.
It won't be as good at predictive modeling for a specific player against a specific team, due to not knowing individual game statistics, only season ones.

## Initial Expectations:

I believe the most stable stats across the seasons will be games played, not necessarily by player but the general distribution. You will have your starters who play most games, your backups who play maybe half the games, and then the bench players who are only getting playing time when multiple people are injured or an opponent is getting blown out. This statistic can't change much because unlike the other statistics, there is a ceiling to the number of games a team can play in a ceiling. While there is a theoretical limit on other statistics, it depends on the team, the era, the seasonal dominance, etc. I also think age will be generally stable because biology tells us men peak at age 25.

In [3]:
missing_counts = nba_data.isnull().sum()

print(missing_counts)

Unnamed: 0              0
player_name             0
team_abbreviation       0
age                     0
player_height           0
player_weight           0
college              1854
country                 0
draft_year              0
draft_round             0
draft_number            0
gp                      0
pts                     0
reb                     0
ast                     0
net_rating              0
oreb_pct                0
dreb_pct                0
usg_pct                 0
ts_pct                  0
ast_pct                 0
season                  0
dtype: int64


```"college"``` is the only default feature will missing values, which makes this dataset nice to work with. These missing values are most likely due to players entering the league from an alternative program, such as the G league, OTE, or international academy.

## Exploratory Data Analysis

Exploratory Data Analysis (EDA) is the first step in data processing and analysis by investigating the data sets and summarizing their main characteristics using visualization, uncovering patterns, spotting irregularities, and finding interesting relationships.

In [5]:
# machine learning imports

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [6]:
# split data 80:20 for training and testing
train_X, val_X, train_y, val_y = train_test_split(X, y, test_size=0.2, random_state=5)

In [7]:
# functions to get mean absolute error based on train test split data and argument:value
def create_model(argument, value):
    model = RandomForestRegressor(
        **{argument: value},
        random_state=5)
    return model

def score_model(model, X_t=train_X, X_v=val_X, y_t=train_y, y_v=val_y):
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    return mean_absolute_error(y_v, preds)

In [8]:
#n_estimators=100
# for estimators in [50, 100, 250, 500]:
#     model= create_model("n_estimators", estimators)
#     print(estimators, score_model(model))
#
# #max_depth=None
# for depth in [5, 10, 15, 20]:
#     model= create_model("max_depth", depth)
#     print(depth, score_model(model))
#
# #min_samples_split=2
# for sample_splits in [2, 5, 10, 20]:
#     model= create_model("min_samples_split", sample_splits)
#     print(sample_splits, score_model(model))
#
# #max_leaf_nodes=None
# for nodes in [50, 100, 250, 500]:
#     model= create_model("max_leaf_nodes", nodes)
#     print(nodes, score_model(model))
#
# #min_samples_leaf=1
# for sample_leaf in [1, 2, 5, 10]:
#     model = create_model("min_samples_leaf", sample_leaf)
#     print(sample_leaf, score_model(model))

Best Individual Argument: min_samples_leaf=2, all other defaults